## [Evaluamos al profesor Marco Cañas Aquí](https://forms.office.com/Pages/ResponsePage.aspx?id=IefhmYRxjkmK_7KtTlPBwkanXIs1i1FEujpsZgO6dXpUREJPV1kxUk1JV1ozTFJIQVNIQjY5WEY3US4u)

# [Vínculo al formulario de google del diagnóstico](https://forms.gle/LXCJMWhBQjfHGEXu8)

Aquí tienes el script de Python que genera el archivo Excel solicitado, realizando la comparación difusa de nombres entre ambos archivos y conservando la estructura del primer archivo con los estudiantes que realizaron el examen.


In [10]:
import pandas as pd
import unicodedata
from rapidfuzz import fuzz

# --- Configuración de rutas (puedes cambiarlas según tus archivos) ---
ruta_primer_archivo = "2_caucasia_final.xlsx"
ruta_segundo_archivo = "4_caucasia_diagnostica_y_final.xls"
ruta_salida = "3_caucasia_final_ordenado.xlsx"

# --- 1. Cargar los datos ---
df_original = pd.read_excel(ruta_primer_archivo, sheet_name=0)
df_referencia = pd.read_excel(ruta_segundo_archivo, sheet_name="P.I")

# --- 2. Limpiar nombres (minúsculas, sin tildes, sin espacios extras) ---
def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return ""
    nombre = str(nombre).lower().strip()
    # Eliminar acentos/tildes
    nombre = unicodedata.normalize("NFKD", nombre).encode("ASCII", "ignore").decode("ASCII")
    # Remover múltiples espacios internos que puedan afectar el match
    nombre = " ".join(nombre.split())
    return nombre

# Construir nombres completos en df_original combinando Primer Nombre y Apellido
# (Se asume que vienen de las columnas originales de la plataforma: 'Student First Name' y 'Student Last Name')
if "Student First Name" in df_original.columns and "Student Last Name" in df_original.columns:
    df_original["Nombres y Apellidos Original"] = (
        df_original["Student First Name"].fillna("").astype(str) + " " + df_original["Student Last Name"].fillna("").astype(str)
    )
elif "Nombres y Apellidos" in df_original.columns:
    df_original["Nombres y Apellidos Original"] = df_original["Nombres y Apellidos"].fillna("").astype(str)
else:
    # Por si acaso la columna ya tuviera otro nombre en tu archivo previo
    df_original["Nombres y Apellidos Original"] = df_original.iloc[:, 0].fillna("").astype(str)

df_original["Nombre original limpio"] = df_original["Nombres y Apellidos Original"].apply(limpiar_nombre)


# Extraer todos los nombres de referencia desde la hoja "P.I" (segundo archivo)
# Empezamos desde el índice 4 (fila 5 de Excel) de la columna de 'RESPUESTAS CORRECTAS' (Columna C / Índice 2)
nombres_referencia = []
for idx, row in df_referencia.iterrows():
    if idx >= 4:  # Índice 4 de pandas corresponde a la fila 5 de Excel
        nombre = row.iloc[2]  # Columna C es el índice 2
        if pd.notna(nombre):
            nombres_referencia.append(str(nombre).strip())

# Crear el DataFrame base con TODOS los estudiantes de referencia para preservar el orden exacto
df_final_base = pd.DataFrame({"Nombres y Apellidos": nombres_referencia})
df_final_base["Nombre referencia limpio"] = df_final_base["Nombres y Apellidos"].apply(limpiar_nombre)

# --- 3. Emparejar usando similitud de cadenas ---
def encontrar_mejor_coincidencia(nombre_ref_limpio, lista_originales_limpios, umbral=60):
    mejor_puntaje = 0
    mejor_nombre = None
    for nombre_orig_limpio in lista_originales_limpios:
        puntaje = fuzz.ratio(nombre_ref_limpio, nombre_orig_limpio)
        if puntaje > mejor_puntaje and puntaje >= umbral:
            mejor_puntaje = puntaje
            mejor_nombre = nombre_orig_limpio
    return mejor_nombre

# Lista de nombres limpios que provienen del archivo de respuestas (df_original)
nombres_orig_limpios = df_original["Nombre original limpio"].unique().tolist()

# Encontrar para cada estudiante del archivo de referencia, cuál es su equivalente en el de respuestas
mapeo_coincidencias = {}
for nombre_ref_limpio in df_final_base["Nombre referencia limpio"].unique():
    coincidencia = encontrar_mejor_coincidencia(nombre_ref_limpio, nombres_orig_limpios)
    if coincidencia:
        mapeo_coincidencias[nombre_ref_limpio] = coincidencia

# Añadir la columna clave de emparejamiento al DataFrame base
df_final_base["Nombre original limpio"] = df_final_base["Nombre referencia limpio"].map(mapeo_coincidencias)

# --- 4. Unir los datos (Left Join) ---
# Eliminamos columnas auxiliares de df_original para evitar duplicados en el resultado
columnas_a_remover = ["Student First Name", "Student Last Name", "Nombres y Apellidos", "Nombres y Apellidos Original"]
df_original_reducido = df_original.drop(columns=[col for col in columnas_a_remover if col in df_original.columns])

# Hacemos el Merge usando la columna de nombres limpios emparejada. 
# Esto mantendrá el 100% de las filas de df_final_base en su estricto orden.
df_resultado = pd.merge(df_final_base, df_original_reducido, on="Nombre original limpio", how="left")

# --- 5. Limpieza Final de Columnas ---
# Eliminamos las columnas temporales que usamos para los cruces de texto
columnas_finales_limpias = [
    col for col in df_resultado.columns 
    if col not in ["Nombre referencia limpio", "Nombre original limpio"]
]
df_resultado = df_resultado[columnas_finales_limpias]

# --- 6. Guardar archivo final ---
df_resultado.to_excel(ruta_salida, index=False)

print(f"Archivo generado exitosamente: {ruta_salida}")
print(f"Total estudiantes requeridos en referencia: {len(df_final_base)}")
print(f"Total filas guardadas en el archivo final: {len(df_resultado)}")

Archivo generado exitosamente: 3_caucasia_final_ordenado.xlsx
Total estudiantes requeridos en referencia: 34
Total filas guardadas en el archivo final: 34


# Explicación del script

### 1. **Carga de archivos**
- Lee el primer archivo `2_bijao_grupo_4_final.xlsx` (con las respuestas).
- Lee la hoja `P.I` del segundo archivo Excel (de referencia).



### 2. **Limpieza de nombres**
- Se crea el nombre completo en el primer archivo: `Student First Name` + `Student Last Name`.
- Se extraen los nombres de la columna `C` de la hoja `P.I` a partir de la **fila 5** (índice 4).
- Se normalizan todos los nombres:
  - Minúsculas
  - Eliminación de tildes (usando `unicodedata`)
  - Eliminación de espacios extra



### 3. **Coincidencia difusa**
- Usa la librería `rapidfuzz` para calcular similitud entre cadenas.
- Cada nombre del primer archivo se compara con todos los nombres de referencia.
- Si la similitud es ≥ **80%**, se considera una coincidencia.



### 4. **Filtrado y creación del archivo final**
- Solo se conservan las filas del primer archivo que tuvieron coincidencia.
- Se crea una nueva columna `Nombre y Apellido` con el nombre exacto tomado del archivo de referencia.
- Se eliminan las columnas auxiliares usadas en el proceso.



### 5. **Guardado**
- El archivo se guarda como `2_bijao_grupo_4_final_corregido.xlsx`.

# Instalación de dependencias

```bash
pip install pandas openpyxl xlrd rapidfuzz
```



## Notas importantes

- El umbral del 80% es configurable; puedes ajustarlo según la calidad de los datos.
- Si un nombre no alcanza el umbral, se excluye del archivo final (su fila queda vacía en el primer archivo, pero no se incluye en la salida porque el requerimiento pide solo estudiantes que realizaron el examen).